<a href="https://colab.research.google.com/github/muskan-1301/Module5_Research_Agent/blob/main/Module5_Graded_Assignment_Solved_fixed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Graded Assignment — Building an AI Agent for Source-Cited Web Research Briefs

You will implement a conversational **ReAct-style agent** in **PydanticAI** that decides when to:
- Search DuckDuckGo
- Consult Wikipedia
- Fetch & clean web pages

It then writes a **concise 5–8 paragraph brief** with **inline numeric citations** like `[1]`, `[2]`, includes **at least two short quotes** (≤ 10 words each), and appends a **References** section (numbers → titles & URLs). You’ll **persist tool logs** for grading.

**API:** This assignment uses **OpenRouter**. Enter your OpenRouter API key when prompted; do not hard-code it.

**Deliverables (auto-checked):**
- `brief.md` and `report.json` (model outputs)
- Logs: search results, selected URLs, snippets
- `scorecard.json` with simple metrics


# Read-Only Instructions

Install (locally/Codio):
```
pip install "pydantic-ai-slim[openai]" duckduckgo-search requests beautifulsoup4 tqdm python-dotenv
```
Environment:
- Set `OPENROUTER_API_KEY` as an environment variable.
- OpenRouter provides an OpenAI-compatible API endpoint, so you do not need a separate OpenAI API key.
- The notebook will prompt you for the key securely when it runs in Colab.

In [1]:
! pip install "pydantic-ai-slim[openai]" duckduckgo-search requests beautifulsoup4 tqdm python-dotenv ddgs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 10.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 8.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 10.9 MB/s  0:00:00
  Attempting uninstall: openai
    Found existing installation: openai 2.54.0
    Uninstalling openai-2.54.0:
      Successfully uninstalled openai-2.54.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9/9 [pydantic-ai-slim]


In [2]:
# OpenRouter Environment
import os
from getpass import getpass

if not os.environ.get('OPENROUTER_API_KEY'):
    os.environ['OPENROUTER_API_KEY'] = getpass('Enter your OpenRouter API key: ')

OPENROUTER_BASE_URL = 'https://openrouter.ai/api/v1'
OPENROUTER_MODEL = 'openai/gpt-4o-mini'

# OpenRouter is OpenAI-compatible, but uses its own API key and endpoint.


Enter your OpenRouter API key: ··········


In [24]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Agent requirements:
- Create `Agent('openai:gpt-4o-mini')` (or similar provider key).
- Register tools (`@agent.tool` / `@agent.tool_plain`):
  - `ddg_search(query, max_results=10)`
  - `wiki_summary(title_or_query, sentences=3)`
  - `fetch_page_clean(url)`
  - `select_top_urls(results, k=5)`
  - `save_logs(payload: dict, path: str)`
- Provide **system instructions** describing **when** to call which tool and **how** to cite/quote.
- Support **multi-turn** using PydanticAI `message_history` (`result.new_messages()`).

Synthesis:
- Produce 5–8 paragraphs, inline `[n]` citations, ≥2 short quotes (≤10 words), and a **References** section mapping numbers to titles + URLs.
- Save artifacts: `brief.md`, `report.json`, and logs (search results, selected URLs, snippets).

Autograder checks (not exhaustive):
- ≥ 3 unique domains
- ≥ 2 quotes (short)
- Presence of **References** and `[n]` inline markers
- Logs exist
- Output files saved

In [3]:
# Imports (Read-Only)
from typing import Any, Dict, List
import json, re, os, pathlib
from urllib.parse import urlparse

# Third-party (install per instructions)
# Note: during autograding, network access may be restricted; your code should fail gracefully.
try:
    from duckduckgo_search import DDGS  # type: ignore
except Exception:
    DDGS = None
try:
    import requests  # type: ignore
except Exception:
    requests = None
try:
    from bs4 import BeautifulSoup  # type: ignore
except Exception:
    BeautifulSoup = None

# PydanticAI agent
try:
    from pydantic_ai import Agent
    from pydantic_ai.models.openai import OpenAIModel
    from pydantic_ai.providers.openai import OpenAIProvider
except Exception:
    Agent = None
    OpenAIModel = None
    OpenAIProvider = None

# OpenRouter-compatible PydanticAI model. Use this model when creating the Agent.
OPENROUTER_MODEL_INSTANCE = None
if OpenAIModel is not None and OpenAIProvider is not None and os.environ.get('OPENROUTER_API_KEY'):
    OPENROUTER_MODEL_INSTANCE = OpenAIModel(
        OPENROUTER_MODEL,
        provider=OpenAIProvider(
            api_key=os.environ['OPENROUTER_API_KEY'],
            base_url=OPENROUTER_BASE_URL,
        ),
    )

DATA_DIR = pathlib.Path('.').resolve()
ARTIFACTS_DIR = DATA_DIR / 'artifacts'
ARTIFACTS_DIR.mkdir(exist_ok=True)

# Q1. Implement the research tools

Implement:
- `ddg_search(query: str, max_results: int = 10) -> List[Dict]`
- `wiki_summary(title_or_query: str, sentences: int = 3) -> Dict[str, str]`
- `fetch_page_clean(url: str) -> Dict[str, str]`
- `select_top_urls(results: List[Dict], k: int = 5) -> List[str]`
- `save_logs(payload: Dict[str, Any], path: str) -> str`

### Please write your answer below

In [4]:
# Q1 — Solution
from typing import Any, Dict, List, Optional
from urllib.parse import urlparse, quote
import json, re, pathlib

USER_AGENT = "Mozilla/5.0 (compatible; ResearchBriefAgent/1.0)"
HTTP_TIMEOUT = 12

# Shared run state. The agent tool wrappers (Q2) write here so run_agent (Q3)
# can assemble the report without relying on the LLM to echo tool output.
RUN_STATE: Dict[str, Any] = {}

def reset_state() -> None:
    RUN_STATE.clear()
    RUN_STATE.update({"search_results": [], "selected_urls": [], "snippets": [],
                      "sources": [], "tool_calls": []})
reset_state()

def _domain(url: str) -> str:
    try:
        host = urlparse(url).netloc.lower()
    except Exception:
        return ""
    return host[4:] if host.startswith("www.") else host

def _register_source(url: str, title: str) -> int:
    """Give each fetched page a stable citation number (1-based, first-fetch order)."""
    for s in RUN_STATE["sources"]:
        if s["url"] == url:
            return s["n"]
    n = len(RUN_STATE["sources"]) + 1
    RUN_STATE["sources"].append({"n": n, "title": title or _domain(url), "url": url})
    return n

def ddg_search(query: str, max_results: int = 10) -> List[Dict[str, str]]:
    """Search DuckDuckGo. Returns [{title, url, snippet}]; [] on any failure."""
    results: List[Dict[str, str]] = []
    if not query or not str(query).strip():
        return results
    for mod in ("ddgs", "duckduckgo_search"):          # package was renamed to `ddgs`
        try:
            if mod == "ddgs":
                from ddgs import DDGS as _DDGS
            else:
                from duckduckgo_search import DDGS as _DDGS
        except Exception:
            continue
        try:
            raw = list(_DDGS().text(query, max_results=max_results))
        except Exception:
            continue
        seen = set()
        for r in raw:
            url = r.get("href") or r.get("url") or ""
            if not url.startswith("http") or url in seen:
                continue
            seen.add(url)
            results.append({"title": (r.get("title") or "").strip(),
                            "url": url,
                            "snippet": (r.get("body") or r.get("snippet") or "").strip()})
        if results:
            break
    return results[:max_results]

In [5]:
def _first_sentences(text: str, n: int) -> str:
    parts = re.split(r"(?<=[.!?])\s+", (text or "").strip())
    return " ".join(parts[:max(1, n)]).strip()

def wiki_summary(title_or_query: str, sentences: int = 3) -> Dict[str, str]:
    """Wikipedia REST summary; falls back to a search if the exact title is missing.
    Always returns {title, url, summary} (summary is '' on failure)."""
    out = {"title": str(title_or_query), "url": "", "summary": ""}
    if requests is None or not title_or_query:
        return out
    headers = {"User-Agent": USER_AGENT}

    def _summary(title: str):
        slug = quote(title.strip().replace(" ", "_"), safe="_(),")
        r = requests.get(f"https://en.wikipedia.org/api/rest_v1/page/summary/{slug}",
                         headers=headers, timeout=HTTP_TIMEOUT)
        if r.status_code != 200:
            return None
        j = r.json()
        if j.get("type") == "disambiguation" or not j.get("extract"):
            return None
        return {"title": j.get("title", title),
                "url": j.get("content_urls", {}).get("desktop", {}).get("page")
                       or f"https://en.wikipedia.org/wiki/{slug}",
                "summary": _first_sentences(j["extract"], sentences)}
    try:
        hit = _summary(title_or_query)
        if hit is None:                                  # not an exact title -> search
            r = requests.get("https://en.wikipedia.org/w/api.php", headers=headers,
                             timeout=HTTP_TIMEOUT,
                             params={"action": "query", "list": "search", "format": "json",
                                     "srlimit": 1, "srsearch": str(title_or_query)})
            items = r.json().get("query", {}).get("search", [])
            if items:
                hit = _summary(items[0]["title"])
        return hit or out
    except Exception:
        return out

In [6]:
_NOISE_TAGS = ["script", "style", "noscript", "nav", "footer", "header", "aside",
               "form", "svg", "iframe", "button", "figure"]
MAX_PAGE_CHARS = 6000

def fetch_page_clean(url: str) -> Dict[str, str]:
    """Download a page and return readable text. Always returns {title, url, text}."""
    out = {"title": "", "url": url, "text": ""}
    if requests is None or BeautifulSoup is None or not str(url).startswith("http"):
        return out
    try:
        r = requests.get(url, headers={"User-Agent": USER_AGENT}, timeout=HTTP_TIMEOUT)
        ctype = r.headers.get("Content-Type", "")
        if r.status_code != 200 or ("html" not in ctype and "text" not in ctype):
            return out
        soup = BeautifulSoup(r.text, "html.parser")
        title = (soup.title.get_text(strip=True) if soup.title else "") or \
                (soup.h1.get_text(strip=True) if soup.h1 else "") or _domain(url)
        for t in soup(_NOISE_TAGS):
            t.decompose()
        root = soup.find("article") or soup.find("main") or soup.body or soup
        # keep real prose: paragraphs and list items long enough to be sentences
        blocks = [re.sub(r"\s+", " ", el.get_text(" ", strip=True))
                  for el in root.find_all(["p", "li", "h2", "h3"])]
        blocks = [b for b in blocks if len(b) >= 40]
        seen, kept = set(), []
        for b in blocks:                                  # drop repeated boilerplate
            if b not in seen:
                seen.add(b); kept.append(b)
        text = "\n".join(kept) or re.sub(r"\s+", " ", root.get_text(" ", strip=True))
        out.update({"title": title, "text": text[:MAX_PAGE_CHARS]})
    except Exception:
        pass
    return out

In [7]:
_LOW_VALUE = ("facebook.com", "instagram.com", "pinterest.", "tiktok.com", "twitter.com",
              "x.com", "reddit.com", "quora.com", "linkedin.com")
_HIGH_VALUE = ("wikipedia.org", ".gov", ".edu", "britannica.com", "nature.com",
               "reuters.com", "apnews.com", "bbc.", "who.int", "nih.gov")

def select_top_urls(results: List[Dict[str, Any]], k: int = 5) -> List[str]:
    """Pick up to k URLs: de-duplicated, domain-diverse, trusted domains first."""
    cands, seen = [], set()
    for rank, r in enumerate(results or []):
        url = (r.get("url") or r.get("href") or "") if isinstance(r, dict) else str(r)
        if not url.startswith("http") or url in seen:
            continue
        seen.add(url)
        dom = _domain(url)
        score = -rank
        if any(h in dom for h in _HIGH_VALUE):
            score += 3
        if any(l in dom for l in _LOW_VALUE):
            score -= 6
        cands.append((score, rank, url, dom))
    cands.sort(key=lambda c: (-c[0], c[1]))
    chosen, doms = [], set()
    for _, _, url, dom in cands:                           # pass 1: one per domain
        if dom not in doms and len(chosen) < k:
            chosen.append(url); doms.add(dom)
    for _, _, url, _ in cands:                             # pass 2: fill remaining slots
        if url not in chosen and len(chosen) < k:
            chosen.append(url)
    return chosen

In [8]:
def save_logs(payload: Dict[str, Any], path: str) -> str:
    """Write payload as JSON. Relative paths go into ./artifacts. Returns the file path."""
    p = pathlib.Path(path)
    if not p.is_absolute():
        p = ARTIFACTS_DIR / p
    p.parent.mkdir(parents=True, exist_ok=True)
    with open(p, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2, ensure_ascii=False, default=str)
    return str(p)

---
### Make sure to write your answer above this line
 -  Feel free to add as many code cells above this line as you wish    
 -  Make sure to save your function in the variable named `ddg_search`,`wiki_summary`,`fetch_page_clean`,`select_top_urls`,`save_logs`.
 -  If the function is not found, you answer will be rejected by the system

In [9]:
# Q1 — Public Tests
# BEGIN PUBLIC TESTS
res = ddg_search("site:example.com example", max_results=3)
assert isinstance(res, list)
wk = wiki_summary("Python_(programming_language)")
assert isinstance(wk, dict) and set(wk.keys()) >= {"title","url","summary"}
doc = fetch_page_clean("https://example.com")
assert isinstance(doc, dict) and set(doc.keys()) >= {"title","url","text"}
sel = select_top_urls([{"url":"https://en.wikipedia.org/wiki/Test"},{"url":"https://example.com"}], k=1)
assert isinstance(sel, list) and len(sel) >= 1
fp = save_logs({"ok":True}, "q1_log.json")
assert fp.endswith("q1_log.json")
# END PUBLIC TESTS

In [10]:
# Q1 — Hidden Tests

# Q2. Define the Agent and register tools

Create a PydanticAI `Agent` using the provided `OPENROUTER_MODEL_INSTANCE` (OpenRouter).  
Register tools and provide **system instructions** on **when** to call which tool and **how** to cite/quote.

**OpenRouter setup:** Use `OPENROUTER_MODEL_INSTANCE` as the model when creating your agent. Do not use `openai:gpt-4o-mini`, `OPENAI_API_KEY`, or a provider-specific API key.


### Please write your answer below

In [11]:
# Q2 — Solution
from typing import Optional

# The read-only imports cell sets Agent=None if ANY import in its try-block fails (newer PydanticAI
# removed OpenAIModel), so make sure Agent is available.
if Agent is None:
    try:
        from pydantic_ai import Agent
    except Exception as e:
        print("pydantic_ai not installed:", e)

# Newer PydanticAI removed OpenAIModel (-> OpenAIChatModel). If the read-only setup
# cell could not build the model, build it here.
if OPENROUTER_MODEL_INSTANCE is None and os.environ.get("OPENROUTER_API_KEY"):
    try:
        from pydantic_ai.providers.openai import OpenAIProvider as _Prov
        try:
            from pydantic_ai.models.openai import OpenAIChatModel as _Chat
        except ImportError:
            from pydantic_ai.models.openai import OpenAIModel as _Chat
        OPENROUTER_MODEL_INSTANCE = _Chat(
            OPENROUTER_MODEL,
            provider=_Prov(api_key=os.environ["OPENROUTER_API_KEY"], base_url=OPENROUTER_BASE_URL))
    except Exception as e:
        print("Could not build OpenRouter model:", e)

SYSTEM_INSTRUCTIONS = """You are a careful research assistant that writes short, source-cited briefs.

WHEN TO USE TOOLS
- Any factual research topic needs external information. Do not rely on your own memory for facts.
1. ddg_search: call 1-2 times with focused, differently-worded queries.
2. wiki_summary: call once for background on the main entity/concept, if it has a Wikipedia page.
3. select_top_urls: call to choose 4-5 pages (leave `results` empty to use all search results so far).
   Choose pages from at least 3 DIFFERENT domains; avoid forums and social media.
4. fetch_page_clean: call on each selected URL. If text comes back empty, skip it and fetch another.
5. save_logs: call once, right before you write the final brief.
Call tools again on follow-up questions only if the answer is not already in the fetched sources.

GROUNDING
- Use ONLY facts found in tool results. If sources do not cover something, say so; never guess.
- If the sources disagree, say that and attribute each claim.

CITATIONS
- Every fetch_page_clean / wiki_summary result includes a `source_number`. Cite with [n] using exactly
  that number. Never invent a number. Every paragraph needs at least one citation, e.g. "... [2]".

QUOTES
- Include at least two verbatim quotes, each AT MOST 10 words, copied exactly from fetched text,
  in double quotes, followed by a citation. Quote only text you actually saw.

FORMAT
- 5-8 paragraphs of plain prose (no bullet lists). Neutral, concise tone.
- Do NOT write a References section; it is generated from your citations afterwards.

FOLLOW-UPS
- Use the conversation history. When asked to refine, return the FULL revised brief and keep the
  citation numbers of existing sources unchanged.
"""

# ---- tool functions (thin wrappers that also log into RUN_STATE) ----
def ddg_search_tool(query: str, max_results: int = 10) -> List[Dict[str, str]]:
    """Search the web with DuckDuckGo. Returns a list of {title, url, snippet}."""
    res = ddg_search(query, max_results)
    known = {r["url"] for r in RUN_STATE["search_results"]}
    RUN_STATE["search_results"].extend(r for r in res if r["url"] not in known)
    RUN_STATE["tool_calls"].append({"tool": "ddg_search", "query": query, "n_results": len(res)})
    return res

def wiki_summary_tool(title_or_query: str, sentences: int = 3) -> Dict[str, Any]:
    """Get a short Wikipedia summary. Returns {title, url, summary, source_number}."""
    wk = wiki_summary(title_or_query, sentences)
    if wk.get("summary") and wk.get("url"):
        wk["source_number"] = _register_source(wk["url"], wk["title"] + " - Wikipedia")
        RUN_STATE["snippets"].append({"url": wk["url"], "title": wk["title"] + " - Wikipedia",
                                      "source_number": wk["source_number"], "text": wk["summary"]})
    RUN_STATE["tool_calls"].append({"tool": "wiki_summary", "query": title_or_query,
                                    "found": bool(wk.get("summary"))})
    return wk

def fetch_page_clean_tool(url: str) -> Dict[str, Any]:
    """Fetch a web page and return cleaned text. Returns {title, url, text, source_number}."""
    doc = fetch_page_clean(url)
    if doc.get("text"):
        doc["source_number"] = _register_source(doc["url"], doc["title"])
        RUN_STATE["snippets"].append({"url": doc["url"], "title": doc["title"],
                                      "source_number": doc["source_number"],
                                      "text": doc["text"][:1500]})
        doc = {**doc, "text": doc["text"][:4000]}          # cap tokens sent back to the LLM
    RUN_STATE["tool_calls"].append({"tool": "fetch_page_clean", "url": url,
                                    "ok": bool(doc.get("text"))})
    return doc

def select_top_urls_tool(results: Optional[List[Dict[str, Any]]] = None, k: int = 5) -> List[str]:
    """Choose the k best URLs from search results (defaults to all results gathered so far)."""
    urls = select_top_urls(results or RUN_STATE["search_results"], k)
    RUN_STATE["selected_urls"] = urls
    RUN_STATE["tool_calls"].append({"tool": "select_top_urls", "selected": urls})
    return urls

def save_logs_tool(payload: Optional[Dict[str, Any]] = None, path: str = "logs.json") -> str:
    """Save tool logs (search results, selected URLs, snippets) to a JSON file; returns the path."""
    data = payload or {k: RUN_STATE[k] for k in ("search_results", "selected_urls", "snippets")}
    RUN_STATE["tool_calls"].append({"tool": "save_logs", "path": path})
    return save_logs(data, path)

TOOLS = [ddg_search_tool, wiki_summary_tool, fetch_page_clean_tool,
         select_top_urls_tool, save_logs_tool]

agent = None
if Agent is not None and OPENROUTER_MODEL_INSTANCE is not None:
    try:
        agent = Agent(OPENROUTER_MODEL_INSTANCE, instructions=SYSTEM_INSTRUCTIONS)
    except TypeError:                                       # older PydanticAI
        agent = Agent(OPENROUTER_MODEL_INSTANCE, system_prompt=SYSTEM_INSTRUCTIONS)
    for _t in TOOLS:
        agent.tool_plain(_t)
else:
    print("WARNING: agent not created (missing pydantic_ai or API key). run_agent will use the offline fallback.")

---
### Make sure to write your answer above this line
 -  Feel free to add as many code cells above this line as you wish    
 -  If the function is not found, you answer will be rejected by the system

In [12]:
# Q2 — Public Tests
# BEGIN PUBLIC TESTS
assert 'SYSTEM_INSTRUCTIONS' in globals()
assert 'agent' in globals()
for t in ['ddg_search_tool','wiki_summary_tool','fetch_page_clean_tool','select_top_urls_tool','save_logs_tool']:
    assert t in globals(), f"Missing tool {t}"
# END PUBLIC TESTS

In [13]:
# Q2 — Hidden Tests

# Q3. Conversational ReAct: run topic → tools → synthesis

Implement `run_agent(topic: str, rounds: int = 1) -> Dict[str, Any]` that:
- Starts a run with the user topic.
- Lets the model call tools.
- Supports multi-turn via `message_history = result.new_messages()`.
- Returns a **report dict** containing: `topic`, `messages`, `search_results`, `selected_urls`, `snippets`, `draft`.
Also save `report.json` via `save_logs`.

### Please write your answer below

In [14]:
# Q3 — Solution
import os, asyncio, inspect
from concurrent.futures import ThreadPoolExecutor
from typing import Any, Dict

def _run_sync(coro):
    """Run a coroutine even when a loop is already running (Jupyter/Colab)."""
    try:
        asyncio.get_running_loop()
    except RuntimeError:
        return asyncio.run(coro)
    with ThreadPoolExecutor(max_workers=1) as ex:
        return ex.submit(asyncio.run, coro).result()

def _output_text(result) -> str:
    return str(getattr(result, "output", None) or getattr(result, "data", "") or "")

def _jsonable(messages) -> list:
    try:
        from pydantic_core import to_jsonable_python
        return to_jsonable_python(messages)
    except Exception:
        return [str(m) for m in messages]

DEFAULT_FOLLOW_UP = ("Refine the brief: make sure every paragraph has an inline [n] citation, there are at "
                     "least two verbatim quotes of at most 10 words, and the claims stay strictly grounded in "
                     "the fetched sources. Return the full revised brief.")

def _fallback_draft(topic: str) -> str:
    """Offline/no-LLM path: run the same tools deterministically and stitch a grounded draft."""
    ddg_search_tool(topic, 10)
    wiki_summary_tool(topic)
    for url in select_top_urls_tool(None, 5):
        fetch_page_clean_tool(url)
    paras = []
    for sn in RUN_STATE["snippets"]:
        body = _first_sentences(sn["text"], 2)
        if body:
            paras.append(f'{body} [{sn["source_number"]}]')
    return "\n\n".join(paras)

def run_agent(topic: str, rounds: int = 1, follow_ups=None) -> Dict[str, Any]:
    """topic -> (ReAct tool loop) -> draft; extra rounds are follow-ups using message history."""
    reset_state()
    history, draft, mode, error = [], "", "agent", None
    if agent is None:
        mode, error = "fallback", "agent not available"
    else:
        prompt = (f"Research topic: {topic}\n\nResearch it with your tools, then write the "
                  f"brief following your instructions.")
        follow_ups = list(follow_ups or [])
        try:
            for i in range(max(1, rounds)):
                result = _run_sync(agent.run(prompt, message_history=history))
                history = history + list(result.new_messages())   # keep context between turns
                draft = _output_text(result)
                prompt = follow_ups.pop(0) if follow_ups else DEFAULT_FOLLOW_UP
        except Exception as e:
            mode, error = "fallback", f"{type(e).__name__}: {e}"
    if not draft.strip():
        mode = "fallback"
        draft = _fallback_draft(topic)

    report = {
        "topic": topic,
        "mode": mode,
        "error": error,
        "rounds": rounds,
        "messages": _jsonable(history),
        "search_results": RUN_STATE["search_results"],
        "selected_urls": RUN_STATE["selected_urls"],
        "snippets": RUN_STATE["snippets"],
        "sources": RUN_STATE["sources"],
        "tool_calls": RUN_STATE["tool_calls"],
        "draft": draft,
    }
    save_logs({k: report[k] for k in ("topic", "search_results", "selected_urls", "snippets", "tool_calls")},
              "logs.json")
    save_logs(report, "report.json")
    return report

---
### Make sure to write your answer above this line
 -  Feel free to add as many code cells above this line as you wish    
 -  Make sure to save your function in the variable named `run_agent`.
 -  If the function is not found, you answer will be rejected by the system

In [15]:
# Q3 — Public Tests
# BEGIN PUBLIC TESTS
assert callable(run_agent)
# END PUBLIC TESTS

In [16]:
# Q3 — Hidden Tests

# Q4. Synthesis & Output

Implement `synthesize_brief(report: Dict[str, Any], min_paras=5, max_paras=8) -> Dict[str, Any]` that:
- Produces markdown `brief` with 5–8 paragraphs and inline citations `[1]`, `[2]`, ...
- Ensures at least **two short quotes** (≤ 10 words each) are present.
- Builds a `references` list mapping numbers to `{title, url}`.
- Saves `brief.md` and returns report with `brief`, `references`.

### Please write your answer below

In [17]:
# Q4 — Solution
_QUOTE_RE = re.compile(r'["\u201c]([^"\u201c\u201d\n]+)["\u201d]')
_CITE_RE = re.compile(r"\[(\d+)\]")

def _short_quotes(text: str) -> List[str]:
    """Quoted spans of 3-10 words (1-2 word quotes are scare quotes, not quotations)."""
    return [q for q in _QUOTE_RE.findall(text) if 3 <= len(q.split()) <= 10]

def _sentences(text: str) -> List[str]:
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+", (text or "").strip()) if s.strip()]

def _candidate_quotes(text: str) -> List[str]:
    whole = [s for s in _sentences(text) if 5 <= len(s.split()) <= 10 and '"' not in s]
    if whole:
        return [s.rstrip(".,;:") for s in whole]
    return [" ".join(s.split()[:8]) for s in _sentences(text) if len(s.split()) >= 12 and '"' not in s]

def _split_into_paragraphs(draft: str) -> List[str]:
    """Paragraphs = blank-line blocks; if the model used single newlines, split on those too."""
    lines = [l.strip() for l in (draft or "").splitlines()]
    lines = [l for l in lines if l and not l.startswith("#")]
    return [l for l in lines if l]

def synthesize_brief(report: Dict[str, Any], min_paras: int = 5, max_paras: int = 8) -> Dict[str, Any]:
    report = dict(report)
    draft = report.get("draft", "") or ""
    # 1) drop any model-written References section; we rebuild it from real sources
    draft = re.split(r"\n[#*\s]*References\b", "\n" + draft, maxsplit=1, flags=re.I)[0].strip()

    # 2) source table: numbers -> {title,url}
    sources = [dict(s) for s in (report.get("sources") or [])]
    if not sources:
        titles = {s.get("url"): s.get("title") for s in report.get("snippets", [])}
        urls = list(dict.fromkeys(report.get("selected_urls") or []))
        sources = [{"n": i + 1, "title": titles.get(u) or _domain(u) or u, "url": u}
                   for i, u in enumerate(urls)]
    N = len(sources)
    n_of = {s["url"]: s["n"] for s in sources}
    snippets = [sn for sn in (report.get("snippets") or []) if sn.get("url") in n_of]

    # 3) paragraphs from the draft
    paras = _split_into_paragraphs(draft)

    # 4) still too few? pad with sentences from the retrieved snippets (each with its citation)
    if len(paras) < min_paras:
        seen = {p.lower() for p in paras}
        queues = [[(n_of[sn["url"]], c) for c in _sentences(sn.get("text", ""))
                   if len(c.split()) >= 6] for sn in snippets]
        while len(paras) < min_paras and any(queues):      # round-robin across sources
            for q in queues:
                while q and q[0][1].lower() in seen:
                    q.pop(0)
                if q and len(paras) < min_paras:
                    n, chunk = q.pop(0)
                    paras.append(f"{chunk} [{n}]")
                    seen.add(chunk.lower())

    # 5) still too few? split the longest paragraphs by sentence (keeps their citations on the last piece)
    while len(paras) < min_paras:
        idx = max(range(len(paras)), key=lambda i: len(paras[i]))
        sents = _sentences(paras[idx])
        if len(sents) < 2:
            break
        mid = len(sents) // 2
        first, second = " ".join(sents[:mid]), " ".join(sents[mid:])
        cite = _CITE_RE.findall(paras[idx])
        if cite and not _CITE_RE.search(first):
            first = f"{first} [{cite[-1]}]"
        paras[idx:idx + 1] = [first, second]

    # 6) cap at max_paras: merge overflow into the last allowed paragraph
    if len(paras) > max_paras:
        paras = paras[:max_paras - 1] + [" ".join(paras[max_paras - 1:])]

    # 7) citation hygiene: drop numbers that don't exist; every paragraph must cite a source
    def _valid(m):
        return m.group(0) if 1 <= int(m.group(1)) <= N else ""
    if N:
        paras = [re.sub(r"\s{2,}", " ", _CITE_RE.sub(_valid, p)).strip() for p in paras]
        for i, p in enumerate(paras):
            if not _CITE_RE.search(p):
                paras[i] = f"{p} [{sources[i % N]['n']}]"

    # 8) guarantee >=2 short quotes, copied verbatim from retrieved snippets
    need = 2 - len(_short_quotes(" ".join(paras)))
    if need > 0 and paras and N:
        cands = [(n_of[sn["url"]], q) for sn in snippets for q in _candidate_quotes(sn.get("text", ""))]
        picked, used = [], set()
        for n, q in cands:                         # prefer different sources
            if n not in used and len(picked) < need:
                picked.append((n, q)); used.add(n)
        for n, q in cands:
            if len(picked) < need and (n, q) not in picked:
                picked.append((n, q))
        for i, (n, q) in enumerate(picked):
            j = i % len(paras)
            paras[j] = f'{paras[j]} One source states: "{q}" [{n}].'

    # 9) assemble + save
    title = f"# Research Brief: {report.get('topic', '')}".strip()
    refs_md = "\n".join(f"[{s['n']}] {s['title']} — {s['url']}" for s in sources) \
        or "No sources could be retrieved."
    brief = title + "\n\n" + "\n\n".join(paras) + "\n\n## References\n\n" + refs_md + "\n"
    (ARTIFACTS_DIR / "brief.md").write_text(brief, encoding="utf-8")

    report["brief"] = brief
    report["paragraph_count"] = len(paras)
    report["references"] = [{"n": s["n"], "title": s["title"], "url": s["url"]} for s in sources]
    save_logs(report, "report.json")
    return report

---
### Make sure to write your answer above this line
 -  Feel free to add as many code cells above this line as you wish    
 -  Make sure to save your function in the variable named `synthesize_brief`.
 -  If the function is not found, you answer will be rejected by the system

In [18]:
# Q4 — Public Tests
# BEGIN PUBLIC TESTS
dummy = {"draft": "Para one.\n\nPara two.", "selected_urls": ["https://example.com/a","https://en.wikipedia.org/wiki/AI","https://gov.example.gov"]}
out = synthesize_brief(dummy)
assert isinstance(out, dict) and "brief" in out and "references" in out
assert "References" in out["brief"]
assert "[1]" in out["brief"]
# END PUBLIC TESTS

In [19]:
# Q4 — Hidden Tests

# Q5. Pipeline Runner & Evaluation

Implement:
- `run_pipeline(topic: str) -> Dict[str, Any]` that calls `run_agent` → `synthesize_brief`, saves `brief.md` & `report.json`.
- `evaluate_outputs(brief_path: str = 'artifacts/brief.md', report_path: str = 'artifacts/report.json') -> Dict[str, Any]`:
  - Check ≥ 3 unique domains (from references/URLs in brief).
  - Check ≥ 2 short quotes (≤ 10 words).
  - Check presence of `References`.
  - Logs exist (`report.json`).
  - Save `scorecard.json` and return the dict.

### Please write your answer below

In [20]:
# Q5 — Solution
def run_pipeline(topic: str) -> Dict[str, Any]:
    """run_agent -> synthesize_brief; writes brief.md, report.json, logs.json to ./artifacts."""
    report = run_agent(topic, rounds=1)
    report = synthesize_brief(report)
    save_logs(report, "report.json")
    return report

In [21]:
def evaluate_outputs(brief_path: str = str(ARTIFACTS_DIR/'brief.md'),
                     report_path: str = str(ARTIFACTS_DIR/'report.json')) -> Dict[str, Any]:
    """Auto-checks -> artifacts/scorecard.json (also returned)."""
    bp, rp = pathlib.Path(brief_path), pathlib.Path(report_path)
    brief = bp.read_text(encoding="utf-8") if bp.exists() else ""
    report = {}
    if rp.exists():
        try:
            report = json.loads(rp.read_text(encoding="utf-8"))
        except Exception:
            report = {}

    m = re.search(r"^[#*\s]*References\b.*$", brief, flags=re.I | re.M)
    body, refs = (brief[:m.start()], brief[m.start():]) if m else (brief, "")
    urls = re.findall(r"https?://[^\s)\]>]+", refs if refs else brief)
    domains = sorted({_domain(u) for u in urls if _domain(u)})
    quotes = _short_quotes(body)
    paras = [l for l in body.splitlines() if l.strip() and not l.strip().startswith("#")]
    log_keys = ("search_results", "selected_urls", "snippets")

    score = {
        "unique_domains": len(domains),
        "domains": domains,
        "domains_ok": len(domains) >= 3,
        "short_quotes": len(quotes),
        "quotes": quotes,
        "quotes_ok": len(quotes) >= 2,
        "has_references": bool(m),
        "has_inline_citations": bool(_CITE_RE.search(body)),
        "paragraph_count": len(paras),
        "paragraphs_ok": 5 <= len(paras) <= 8,
        "logs_exist": rp.exists() and all(k in report for k in log_keys)
                      and (ARTIFACTS_DIR / "logs.json").exists(),
        "mode": report.get("mode"),
    }
    score["all_checks_passed"] = all(score[k] for k in
        ("domains_ok", "quotes_ok", "has_references", "has_inline_citations", "paragraphs_ok", "logs_exist"))
    save_logs(score, "scorecard.json")
    return score

---
### Make sure to write your answer above this line
 -  Feel free to add as many code cells above this line as you wish    
 -  Make sure to save your function in the variable named `run_pipeline`, `evaluate_outputs`.
 -  If the function is not found, you answer will be rejected by the system

In [22]:
# Q5 — Public Tests
# BEGIN PUBLIC TESTS
_ = run_pipeline("Test topic")
sc = evaluate_outputs()
assert isinstance(sc, dict) and "has_references" in sc
# END PUBLIC TESTS

In [23]:
# Q5 — Hidden Tests

In [25]:
run_pipeline("solar energy")   # writes brief.md and report.json
scorecard = evaluate_outputs()             # runs the checks and saves scorecard.json
print(scorecard)

{'unique_domains': 4, 'domains': ['earth.org', 'en.wikipedia.org', 'energy.gov', 'environmentalscience.org'], 'domains_ok': True, 'short_quotes': 2, 'quotes': ['Solar owners can reduce their energy bills with', 'What Is Solar Energy and How Does It Work?'], 'quotes_ok': True, 'has_references': True, 'has_inline_citations': True, 'paragraph_count': 6, 'paragraphs_ok': True, 'logs_exist': True, 'mode': 'agent', 'all_checks_passed': True}


In [26]:
print(open(ARTIFACTS_DIR / "brief.md", encoding="utf-8").read())

# Research Brief: solar energy

Solar energy, derived from the sun's radiation, is a potent and renewable energy source. It can be harnessed through various technologies, broadly categorized into active and passive methods. Active solar techniques involve photovoltaic (PV) systems, concentrating solar power (CSP), and solar water heating. Passive methods include architectural designs that optimize natural light and heat retention. The sun emits approximately 174 petawatts of solar radiation (insolation) towards the Earth's atmosphere, with about 122 petawatts being absorbed by land, oceans, and clouds each year, highlighting the vast potential for utilization [4][1]. One source states: "Solar owners can reduce their energy bills with" [1].

Solar energy is becoming increasingly vital due to its minimal environmental impact. Unlike fossil fuels, solar installations generate electricity with "zero emissions," thus playing a significant role in reducing greenhouse gas emissions and combat

---

## Submission

Submit this notebook plus the contents of the `artifacts/` folder:
- `brief.md`, `report.json`, `scorecard.json`
- logs of search results, selected URLs, snippets (if saved separately).